# Approach 4 — Open-Form Segmentation

## What this approach does and why it is different

The three earlier approaches (rules-based, K-Means archetype clustering, composite
scoring) all start with the **Archetypes Progress framework** as a fixed grid and either
map practices into it or score them against it. That is intentional — the framework has
real strategic value.

This notebook does something different: it **removes the framework entirely** and asks
the data alone to reveal its natural groupings.

### Why bother?

The pre-defined 4×4 matrix encodes assumptions — that Size and Model are the two primary
axes of variation, and that each has exactly four tiers. Those assumptions are reasonable
but they may not be the *only* structure in the data. Open-form segmentation can surface:

- **Cross-cutting clusters** — e.g. a group of practices that are "mid-size NHS but
  unusually high-throughput" that don't fit cleanly anywhere in the 4×4 grid
- **The real dimensionality** — how many genuinely distinct practice *types* exist?
  Four per axis (16 total) is a human design choice; the data may suggest fewer or more
- **Blind spots** — groups with a distinctive operational signature that the archetype
  framework, by construction, cannot distinguish

### How it works

1. **Feature engineering** — 8 features are constructed that capture four dimensions:
   physical scale, commercial orientation, operational intensity and staffing model
2. **Normalisation** — all features are z-scored so no single dimension dominates by virtue of units
3. **Optimal k selection** — both the elbow (inertia) curve and the silhouette score are
   computed for k = 2 … 10; the silhouette peak indicates the most internally coherent
   solution
4. **K-Means clustering** — fit at the chosen k with multiple restarts for stability
5. **PCA visualisation** — reduce to 2 principal components for a scatter view of the
   cluster structure
6. **Centroid profiling and labelling** — each cluster centroid is translated into a
   human-readable label using a rule-based profile interpreter that reads the centroid's
   position on each feature dimension
7. **Archetype cross-tab** — compare discovered segments against the rules-based
   archetype labels to understand the relationship between the two views

### Key difference from `02_clustering.ipynb`

| | Archetype clustering (02) | Open segmentation (06) |
|---|---|---|
| **k per axis** | Fixed at 4 | Chosen by silhouette score |
| **Axes** | Size + Model separately | Single joint space on 8 features |
| **Labels** | Must map to archetype names | Discovered from centroid profile |
| **Output** | 4×4 grid cell | Single segment label |
| **Goal** | Reproduce the framework from data | Let data reveal its own structure |

---
## 1 · Setup and feature engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples
import matplotlib.cm as cm

warnings.filterwarnings('ignore')

PALETTE = 'tab10'
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120,
                     'axes.spines.top': False,
                     'axes.spines.right': False})

# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('master.csv')
print(f'Loaded {len(df):,} practices')

# ── Feature engineering ───────────────────────────────────────────────────────
NHS_VALUE_PER_UDA = 28.0

df['nhs_income_est']  = df['uda'].fillna(0) * NHS_VALUE_PER_UDA
df['total_income_est']= df['privateincome'].fillna(0) + df['nhs_income_est']
df['private_income_pct'] = np.where(
    df['total_income_est'] > 0,
    df['privateincome'].fillna(0) / df['total_income_est'] * 100, np.nan)

df['total_treatment_items'] = df['nooftreatmentitems'].fillna(0)
df['items_per_surgery'] = (
    df['total_treatment_items'] / df['numberofsurgeries'].replace(0, np.nan))
df['income_per_surgery'] = (
    df['total_income_est'] / df['numberofsurgeries'].replace(0, np.nan))
df['nurse_to_dentist_ratio'] = (
    df['position_dental_nurse'] / df['position_dentist'].replace(0, np.nan))
df['hours_per_dentist'] = (
    df['contractualhours_dentist'] / df['position_dentist'].replace(0, np.nan))

# ── Feature matrix ────────────────────────────────────────────────────────────
# Eight features across four dimensions:
#   Physical scale:       numberofsurgeries, income_per_surgery
#   Commercial mix:       private_income_pct, nhs_income_est
#   Operational intensity: items_per_surgery, total_treatment_items
#   Staffing model:       nurse_to_dentist_ratio, position_hygienist

FEATURES = [
    'numberofsurgeries',
    'income_per_surgery',
    'private_income_pct',
    'nhs_income_est',
    'items_per_surgery',
    'total_treatment_items',
    'nurse_to_dentist_ratio',
    'position_hygienist',
]

FEATURE_LABELS = [
    'Surgeries',
    'Income / surgery',
    'Private income %',
    'NHS income (est.)',
    'Items / surgery',
    'Total items',
    'Nurse : Dentist ratio',
    'Hygienists',
]

feat_df = df[FEATURES].dropna()
valid_idx = feat_df.index
print(f'Practices with complete features: {len(valid_idx)}')

scaler = StandardScaler()
X = scaler.fit_transform(feat_df)

print('\nFeature summary (raw):')
print(feat_df.describe().round(1).to_string())

#### Interpretation — Feature space

These 8 features were chosen because they span the four dimensions that most distinguish
dental practices operationally. Crucially they include **ratio features** (`items_per_surgery`,
`income_per_surgery`, `nurse_to_dentist_ratio`) not used in the archetype clustering —
these capture *how efficiently* a practice uses its resources, not just *how large* it is.

**`StandardScaler` normalisation** converts every feature to mean 0, standard deviation 1
before clustering, ensuring that large-magnitude features like `nhs_income_est` (£thousands)
don't dominate over binary-scale features like `position_hygienist` (0–2).

---
## 2 · Choosing the number of segments

In [ ]:
K_RANGE = range(2, 11)
inertias, sil_scores = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Choosing optimal k — Elbow + Silhouette', fontsize=13, fontweight='bold')

ks = list(K_RANGE)
ax1.plot(ks, inertias, marker='o', color='#4C72B0', lw=2)
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('Inertia (within-cluster sum of squares)')
ax1.set_title('Elbow curve')
ax1.set_xticks(ks)

ax2.plot(ks, sil_scores, marker='o', color='#DD8452', lw=2)
best_k = ks[int(np.argmax(sil_scores))]
ax2.axvline(best_k, color='red', lw=1.5, ls='--', label=f'Best k = {best_k}')
ax2.set_xlabel('Number of clusters (k)')
ax2.set_ylabel('Silhouette score (higher = better)')
ax2.set_title('Silhouette score')
ax2.set_xticks(ks)
ax2.legend()

plt.tight_layout()
plt.show()

print(f'Silhouette scores: {dict(zip(ks, [round(s,3) for s in sil_scores]))}')
print(f'Suggested optimal k = {best_k} (highest silhouette)')

#### Interpretation — Elbow and silhouette

**Elbow curve**: look for the "kink" where adding another cluster gives diminishing
reduction in inertia. A sharp elbow at k=5 means 5 clusters explain much of the
within-group variance; the 6th cluster adds little new structure.

**Silhouette score** (range: −1 to +1) measures how well each practice fits its own
cluster vs its nearest neighbour cluster. A peak at k=5 means that k=5 produces the
most internally coherent and well-separated segments. This is the primary guide.

If the two methods disagree (elbow says 4, silhouette says 6), prefer the silhouette —
it is a direct measure of cluster *quality*, while inertia alone doesn't penalise for
creating clusters that overlap.

---
## 3 · Fit final segmentation model

In [ ]:
# Use the silhouette-optimal k; override here if domain knowledge suggests otherwise
K_FINAL = best_k

km_final = KMeans(n_clusters=K_FINAL, n_init=30, random_state=42)
cluster_raw = km_final.fit_predict(X)

df.loc[valid_idx, 'segment_id'] = cluster_raw
df['segment_id'] = df['segment_id'].astype('Int64')

print(f'Final model: k = {K_FINAL}')
print(f'Cluster sizes:\n{pd.Series(cluster_raw).value_counts().sort_index().to_string()}')
print(f'\nFinal inertia: {km_final.inertia_:.1f}')
print(f'Final silhouette score: {silhouette_score(X, cluster_raw):.3f}')

---
## 4 · PCA scatter — cluster structure in 2D

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

df.loc[valid_idx, 'pca1'] = X_pca[:, 0]
df.loc[valid_idx, 'pca2'] = X_pca[:, 1]

var_exp = pca.explained_variance_ratio_ * 100

# Build loadings annotation
loadings = pd.DataFrame(pca.components_.T, index=FEATURE_LABELS,
                        columns=['PC1', 'PC2'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('4 · PCA Scatter — Natural Cluster Structure', fontsize=13, fontweight='bold')

# Scatter coloured by cluster
cmap = plt.get_cmap(PALETTE)
plot_df = df.loc[valid_idx].copy()
for cid in sorted(plot_df['segment_id'].dropna().unique()):
    sub = plot_df[plot_df['segment_id'] == cid]
    axes[0].scatter(sub['pca1'], sub['pca2'],
                    label=f'Cluster {int(cid)}',
                    color=cmap(int(cid) / K_FINAL),
                    alpha=0.55, s=40, edgecolors='none')
# Mark centroids in PCA space
pca_centroids = pca.transform(km_final.cluster_centers_)
for cid, (cx, cy) in enumerate(pca_centroids):
    axes[0].scatter(cx, cy, color=cmap(cid / K_FINAL),
                    s=200, marker='*', edgecolors='black', linewidths=0.8, zorder=5)
    axes[0].annotate(str(cid), (cx, cy), textcoords='offset points',
                     xytext=(6, 4), fontsize=9, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({var_exp[0]:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({var_exp[1]:.1f}% variance)')
axes[0].set_title('Clusters in PCA space (★ = centroid)')
axes[0].legend(fontsize=9, loc='upper right')

# Loadings heatmap
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=axes[1], vmin=-1, vmax=1, annot_kws={'size': 9})
axes[1].set_title('PC loadings — what each axis captures')

plt.tight_layout()
plt.show()

print(f'PC1 explains {var_exp[0]:.1f}% | PC2 explains {var_exp[1]:.1f}% | '
      f'Combined {sum(var_exp[:2]):.1f}%')

#### Interpretation — PCA scatter and loadings

**The scatter** shows whether clusters are spatially separated or overlapping. Well-separated
clusters (clear gaps between coloured groups) indicate that the K-Means solution is robust —
the groups are genuinely distinct in the 8-dimensional feature space.

**The loadings heatmap** reveals what PC1 and PC2 actually measure:
- High absolute loadings on PC1 from multiple features mean PC1 is a general "scale" axis
  — practices spread left-to-right from small/low-volume to large/high-volume.
- If PC2 loads heavily on `private_income_pct` and `position_hygienist`, it is a
  "commercial model" axis separating NHS-heavy from private-oriented practices.

Knowing what the axes mean helps interpret *why* clusters sit where they do — a cluster
in the top-right is large-scale AND private-oriented; one in the bottom-left is small-scale
AND NHS-led.

---
## 5 · Centroid profiles and segment labelling

In [ ]:
# ── Centroid profile in original (unscaled) units ────────────────────────────
centroids_raw = scaler.inverse_transform(km_final.cluster_centers_)
centroid_df = pd.DataFrame(centroids_raw, columns=FEATURES)
centroid_df.index.name = 'cluster'

# Portfolio medians for relative comparison
port_medians = feat_df.median()

print('Cluster centroids (original units):')
print(centroid_df.round(1).to_string())

print('\nPortfolio medians:')
print(port_medians.round(1).to_string())

In [ ]:
# ── Auto-label each cluster from its centroid profile ────────────────────────
def label_cluster(centroid, medians):
    """
    Generate a descriptive segment label from centroid values relative
    to portfolio medians.  Returns a short name and a one-line description.
    """
    surg       = centroid['numberofsurgeries']
    priv_pct   = centroid['private_income_pct']
    items_rate = centroid['items_per_surgery']
    hygienist  = centroid['position_hygienist']
    nhs_inc    = centroid['nhs_income_est']
    inc_surg   = centroid['income_per_surgery']
    n2d        = centroid['nurse_to_dentist_ratio']

    m_surg     = medians['numberofsurgeries']
    m_priv     = medians['private_income_pct']
    m_items    = medians['items_per_surgery']
    m_inc_surg = medians['income_per_surgery']

    # ── Size tag ──────────────────────────────────────────────────────────────
    if surg < m_surg * 0.75:
        size_tag = 'Compact'
    elif surg > m_surg * 1.25:
        size_tag = 'Large-Scale'
    else:
        size_tag = 'Mid-Size'

    # ── Commercial model tag ──────────────────────────────────────────────────
    if priv_pct >= 60:
        model_tag, model_desc = 'Private-Led', 'high private income share'
    elif priv_pct <= 40:
        model_tag, model_desc = 'NHS-Led', 'high NHS contract dependency'
    else:
        model_tag, model_desc = 'Balanced', 'balanced NHS/private split'

    # ── Productivity tag ──────────────────────────────────────────────────────
    if items_rate > m_items * 1.20:
        prod_tag = 'High-Throughput'
    elif items_rate < m_items * 0.80:
        prod_tag = 'Lower-Throughput'
    else:
        prod_tag = 'Standard-Throughput'

    # ── Yield tag ─────────────────────────────────────────────────────────────
    if inc_surg > m_inc_surg * 1.20:
        yield_tag = 'high income yield per surgery'
    elif inc_surg < m_inc_surg * 0.80:
        yield_tag = 'lower income yield per surgery'
    else:
        yield_tag = 'average income yield per surgery'

    # ── Staffing tag ──────────────────────────────────────────────────────────
    hyg_tag = 'hygienist-enabled' if hygienist > 0.5 else 'no hygienist'

    name = f'{size_tag} · {model_tag} · {prod_tag}'
    desc = (
        f'{size_tag.lower()} practice footprint, '
        f'{model_desc}, '
        f'{prod_tag.lower().replace("-"," ")} per chair, '
        f'{yield_tag}, '
        f'{hyg_tag}'
    )
    return name, desc


labels, descs = {}, {}
for cid in range(K_FINAL):
    name, desc = label_cluster(centroid_df.loc[cid], port_medians)
    labels[cid] = name
    descs[cid]  = desc

print('Generated segment labels:')
for cid, (name, desc) in enumerate(zip(labels.values(), descs.values())):
    n_practices = (df['segment_id'] == cid).sum()
    print(f'  Cluster {cid} (n={n_practices:3d}):  {name}')
    print(f'           {desc}')
    print()

df['segment_label'] = df['segment_id'].map(labels)

#### Interpretation — Auto-generated labels

The label function reads each centroid's position on five dimensions relative to the
portfolio median:

| Dimension | Tag examples |
|---|---|
| Physical scale | Compact / Mid-Size / Large-Scale |
| Commercial model | NHS-Led / Balanced / Private-Led |
| Throughput intensity | Lower-Throughput / Standard / High-Throughput |
| Income yield | lower / average / high income yield per surgery |
| Specialist capability | hygienist-enabled / no hygienist |

**These labels are descriptive, not prescriptive.** They tell you *what the cluster looks
like today*, not what it should aspire to be. A "Compact · NHS-Led · Lower-Throughput"
segment is not inherently bad — it may be a stable community NHS service with low
commercial ambition, and that may be entirely appropriate for its location and contract.

> **Tip for real data:** Review the auto-generated labels and override any that
> don't match the business intuition of someone who knows the portfolio. The code
> exposes `labels` as a plain dictionary — update it directly.

---
## 6 · Radar profiles — how each segment scores on every feature

In [ ]:
# Normalise centroids to 0-1 range (min-max across clusters) for radar
radar_df = centroid_df.copy()
for col in FEATURES:
    col_min, col_max = radar_df[col].min(), radar_df[col].max()
    radar_df[col] = (radar_df[col] - col_min) / (col_max - col_min + 1e-9)

N_FEATURES = len(FEATURES)
angles = np.linspace(0, 2 * np.pi, N_FEATURES, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

n_cols = min(K_FINAL, 3)
n_rows = int(np.ceil(K_FINAL / n_cols))
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(5 * n_cols, 4.5 * n_rows),
                         subplot_kw=dict(polar=True))
fig.suptitle('6 · Segment Radar Profiles (0 = portfolio min, 1 = portfolio max)',
             fontsize=13, fontweight='bold')

axes_flat = np.array(axes).flatten()
cmap = plt.get_cmap(PALETTE)

for cid in range(K_FINAL):
    ax = axes_flat[cid]
    values = radar_df.loc[cid].tolist() + [radar_df.loc[cid].tolist()[0]]
    color = cmap(cid / K_FINAL)
    n_prac = int((df['segment_id'] == cid).sum())

    ax.plot(angles, values, color=color, lw=2)
    ax.fill(angles, values, color=color, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(FEATURE_LABELS, fontsize=8)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['', '', ''], fontsize=7)
    ax.set_title(f'Cluster {cid}  (n={n_prac})\n{labels[cid]}',
                 fontsize=9, fontweight='bold', pad=12)

# Hide unused axes
for ax in axes_flat[K_FINAL:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

#### Interpretation — Radar profiles

Each polygon represents one cluster's "fingerprint" across the 8 features. The scale
is **relative across clusters** (0 = the cluster with the lowest value on that feature,
1 = the highest), so you can compare shape, not just magnitude.

**What to look for:**
- A **large filled area** = a high-resource cluster (large, busy, well-staffed)
- A cluster that scores high on `Private income %` but low on `NHS income` has a clear
  private orientation — look at whether `income_per_surgery` is also high (healthy) or
  low (under-charging for private)
- A cluster high on `Items / surgery` but low on `Income / surgery` is generating volume
  without proportionate revenue — a case-mix pricing opportunity
- A cluster with a large `Nurse:Dentist ratio` spike is staffed for clinical throughput —
  compare this to its `Surgeries` and `Items / surgery` to see if that capacity is being used

---
## 7 · Silhouette analysis — how confidently assigned is each practice?

In [ ]:
sil_vals   = silhouette_samples(X, cluster_raw)
df.loc[valid_idx, 'silhouette_score'] = sil_vals

fig, ax = plt.subplots(figsize=(10, max(5, K_FINAL * 1.2)))
y_lower = 10
cmap = plt.get_cmap(PALETTE)

for cid in range(K_FINAL):
    cid_sil = np.sort(sil_vals[cluster_raw == cid])
    size_c  = len(cid_sil)
    y_upper = y_lower + size_c

    color = cmap(cid / K_FINAL)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cid_sil,
                     facecolor=color, edgecolor='none', alpha=0.8)
    ax.text(-0.03, (y_lower + y_upper) / 2, str(cid), fontsize=9,
            fontweight='bold', va='center')
    y_lower = y_upper + 10

mean_sil = sil_vals.mean()
ax.axvline(mean_sil, color='red', lw=1.5, ls='--',
           label=f'Mean silhouette = {mean_sil:.3f}')
ax.set_xlabel('Silhouette coefficient')
ax.set_title('7 · Per-practice silhouette scores by cluster', fontweight='bold')
ax.set_yticks([])
ax.legend()
plt.tight_layout()
plt.show()

# Flag practices with low silhouette (borderline assignment)
LOW_SIL_THRESHOLD = 0.10
df.loc[valid_idx, 'segment_borderline'] = sil_vals < LOW_SIL_THRESHOLD
n_borderline = df['segment_borderline'].sum()
print(f'Practices with silhouette < {LOW_SIL_THRESHOLD} (borderline): '
      f'{n_borderline} ({n_borderline/len(df)*100:.1f}%)')
print('\nAverage silhouette per cluster:')
for cid in range(K_FINAL):
    mean_c = sil_vals[cluster_raw == cid].mean()
    print(f'  Cluster {cid} — {labels[cid]}: {mean_c:.3f}')

#### Interpretation — Silhouette scores

The silhouette coefficient for a single practice measures:
- How similar it is to practices **in its own cluster** (cohesion)
- How different it is from practices **in the nearest other cluster** (separation)

A score near **+1** = clearly belongs to its cluster.
A score near **0** = sitting on the border between two clusters.
A score near **−1** = may have been assigned to the wrong cluster.

**What this means at practice level:**
- Practices with low silhouette scores are **genuinely ambiguous** — they are neither
  clearly one segment nor the other. These are exactly the practices that would be
  "blend" cases in the scoring approach (Notebook 03).
- A cluster where most silhouette bars are **short and left-leaning** (close to 0) is
  not a well-defined segment — consider whether merging it with a neighbour would
  produce cleaner groups.
- A cluster where scores cluster around **0.4–0.7** is a strong, distinctive segment
  that is genuinely different from the rest of the portfolio.

---
## 8 · Segment summary table

In [ ]:
summary_cols = {
    'segment_label':        'Segment',
    'practicekey':          'n',
    'numberofsurgeries':    'Avg surgeries',
    'unique_staff_ids':     'Avg staff',
    'private_income_pct':   'Private % (avg)',
    'privateincome':        'Avg private income (£)',
    'nhs_income_est':       'Avg NHS income (£)',
    'items_per_surgery':    'Items / surgery',
    'income_per_surgery':   'Income / surgery (£)',
    'nurse_to_dentist_ratio':'Nurse:Dentist ratio',
    'position_hygienist':   'Avg hygienists',
    'silhouette_score':     'Mean silhouette',
}

summary = (
    df.groupby('segment_label')
    .agg(
        n=('practicekey', 'count'),
        avg_surgeries=('numberofsurgeries', 'mean'),
        avg_staff=('unique_staff_ids', 'mean'),
        private_pct=('private_income_pct', 'mean'),
        avg_private_income=('privateincome', 'mean'),
        avg_nhs_income=('nhs_income_est', 'mean'),
        items_per_surg=('items_per_surgery', 'mean'),
        income_per_surg=('income_per_surgery', 'mean'),
        nurse_dent=('nurse_to_dentist_ratio', 'mean'),
        avg_hygienists=('position_hygienist', 'mean'),
        mean_sil=('silhouette_score', 'mean'),
    )
    .round(2)
)

summary.columns = ['n', 'Surgeries', 'Staff', 'Private %',
                   'Priv income £', 'NHS income £', 'Items/surg',
                   'Income/surg £', 'N:D ratio', 'Hygienists', 'Silhouette']

display(summary)

#### Interpretation — Summary table

This table is the primary **practice benchmarking reference** for the open segmentation.
For any individual practice, identify its segment label (from `df['segment_label']` or
the output CSV) and compare its raw metrics to the segment averages above.

**High-value comparison points:**
- `Income/surg £` vs segment average: is the practice extracting the revenue its segment
  peers achieve per chair? If not, it's under-performing commercially within its natural peer group.
- `Items/surg` vs segment average: is the practice filling its chairs to the level its
  segment peers do? A low-items practice in a High-Throughput segment likely has a
  scheduling or demand-generation problem.
- `N:D ratio` vs segment average: outlier practices in a segment may have a staffing
  model mismatch — either over-nursing (cost inefficiency) or under-nursing (clinical risk).

---
## 9 · Relationship to archetype framework

In [ ]:
# Load rules-based archetype labels
try:
    arc = pd.read_csv('archetypes_rules.csv')[['practicekey', 'archetype_size',
                                                'archetype_model', 'archetype_rules']]
    df_arc = df.merge(arc, on='practicekey', how='left')
    HAS_ARC = df_arc['archetype_size'].notna().sum() > 0
except FileNotFoundError:
    HAS_ARC = False
    print('archetypes_rules.csv not found — run 01_rules_based.ipynb first.')

if HAS_ARC:
    # Cross-tab: open segment vs archetype size
    ct_size = pd.crosstab(df_arc['segment_label'], df_arc['archetype_size'],
                          margins=True, margins_name='Total')
    print('Segment vs Archetype Size:')
    display(ct_size)

    # Cross-tab: open segment vs archetype model
    ct_model = pd.crosstab(df_arc['segment_label'], df_arc['archetype_model'],
                           margins=True, margins_name='Total')
    print('\nSegment vs Archetype Model:')
    display(ct_model)

In [ ]:
if HAS_ARC:
    # Heatmap: what % of each archetype_rules cell falls into each open segment?
    ct_full = pd.crosstab(df_arc['segment_label'], df_arc['archetype_rules'])
    ct_full_pct = ct_full.div(ct_full.sum(axis=0), axis=1) * 100

    fig, ax = plt.subplots(figsize=(max(8, ct_full_pct.shape[1] * 1.2),
                                    max(5, ct_full_pct.shape[0] * 0.8)))
    sns.heatmap(ct_full_pct.round(1), annot=True, fmt='.0f', cmap='Blues',
                linewidths=0.5, ax=ax, cbar_kws={'label': '% of archetype cell'})
    ax.set_xlabel('Archetype (Size × Model)')
    ax.set_ylabel('Open segment')
    ax.set_title('9 · % of each archetype cell captured by each open segment',
                 fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

#### Interpretation — Segment vs archetype cross-tab

This comparison reveals the **relationship between the two frameworks**:

- **One-to-one alignment** (a single archetype cell maps almost entirely to one segment):
  the open segmentation and the archetype framework are telling the same story — the
  archetype definition is a good approximation of the natural data structure.
- **Many-to-one** (multiple archetype cells collapse into one segment): the open
  segmentation doesn't distinguish between those archetype cells because the data
  features don't separate them. This is a signal that the archetype boundary may be
  over-specified for the available data.
- **One-to-many** (one archetype cell splits across multiple segments): the open
  segmentation is finding sub-types *within* an archetype definition — real heterogeneity
  that the framework treats as homogeneous. These sub-groups may warrant different
  operational or staffing responses.

---
## 10 · Segment distribution by region

In [ ]:
reg_seg = pd.crosstab(df['region'], df['segment_label'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(12, 5))
reg_seg.plot(kind='bar', stacked=True, colormap=PALETTE,
             edgecolor='white', linewidth=0.5, ax=ax)
ax.set_xlabel('Region')
ax.set_ylabel('% of practices in region')
ax.set_title('10 · Segment mix by region', fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
ax.legend(title='Segment', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

print('Segment count by region:')
print(pd.crosstab(df['region'], df['segment_label']).to_string())

#### Interpretation — Regional segment mix

Regional variation in segment distribution answers the question: **are certain practice
types concentrated in particular geographies?**

- A region dominated by one segment is operationally homogeneous — a single staffing
  template or commercial playbook is likely sufficient.
- A region with a diverse segment mix requires differentiated approaches — the same
  initiative (e.g. a private recall programme rollout) will land very differently in a
  "Compact · NHS-Led" practice vs a "Large-Scale · Private-Led" one.
- If a segment appears in only one or two regions, investigate whether it is a genuine
  market phenomenon (demographics, competition) or an artefact (e.g. a cluster of
  recently acquired practices all in the same geography).

---
## 11 · Save output

In [ ]:
out_cols = ['practicekey', 'practicename', 'region',
            'segment_id', 'segment_label', 'segment_borderline',
            'silhouette_score',
            'numberofsurgeries', 'private_income_pct',
            'items_per_surgery', 'income_per_surgery',
            'nurse_to_dentist_ratio', 'position_hygienist']

out = df[[c for c in out_cols if c in df.columns]].copy()
out.to_csv('archetypes_open_segmentation.csv', index=False)
print(f'Saved {len(out):,} rows to archetypes_open_segmentation.csv')

# Print final label distribution
print('\nFinal segment distribution:')
print(df['segment_label'].value_counts().to_string())